# **Esame del 10 settembre 2024**

- Non si possono consultare libri, note, ed ogni altro materiale o persone durante
l’esame ad eccezione delle funzioni Python fornite.

- Risolvere i seguenti esercizi con l’ausilio di Python.

- La durata del compito è di 90 minuti.

- Questo esame ha 3 domande, per un totale di 30/30 punti + 1 extra.

- Svolgere gli esercizi marcati con **[T]** su fogli protocollo, indicando: nome, cognome, codice persona
e data

# **Esercizio 1 (punti 10)**

Si consideri il seguente problema algebrico: data $A \in \mathbb{R}^{n\times n}$ e $b \in \mathbb{R}^n$ trovare $x\in \mathbb{R}^n$ tale che
$$
A x = b
$$
dove $A$ e $b$ sono dati da
$${A}=\left[\begin{array}{rrrrrrr}
9 & -3 & 1 &  &  & & \\
-3 & 9 & -3 & 1 &  & & \\
1 & -3 & 9 & -3 & 1 & & \\
& 1 & -3 & 9 & -3 & 1 &\\
& & 1 & -3 & 9 & -3 & 1 \\
& & & 1 & -3 & 9 & -3  \\
& & & & 1 & -3 & 9  \\
\end{array}\right]
\quad\quad {b}=\left[\begin{array}{c}7\\4\\5\\5\\5\\4\\7\end{array}\right].$$


#### **(1a) (2 punti) [T]**  

Uno schema iterativo per la soluzione di sistemi lineari si può scrivere nella seguente forma generale: dato $x^0 \in \mathbb{R}^n$ calcolo la soluzione tentativo al passo successivo $x^{k+1}\in \mathbb{R}^n$ come
$$
x^{k+1} = B x^{k} + g.
$$
Enunciare la condizione necessaria e sufficiente per la sua convergenza.

#### **(1b) (3 punti) [T]**  

Illustrare i casi particolari del metodo di Jacobi e di Gauss-Seidel per la risoluzione del problema precedente, enunciarne quindi le condizioni di convergenza (sufficienti e necessarie-e-sufficienti).

#### **(1c) (2 punti) [P]**  

Verificare che per il problema in considerazione le condizioni sopra enunciate siano soddisfatte (verificare una condizione a scelta per metodo), utilizzando le funzioni `Jacobi_Bc` e `GS_Bc` per costruire le matrici di iterazione. Per il calcolo degli autovalori si può usare la funzione `eigvals` del pacchetto `scipy.linalg`.

In [ ]:
import numpy as np

from scipy.linalg import eigvals

from functions import Jacobi_Bc, GS_Bc

#### **(1d) (3 punti) [P+T]**

Sia $x^*\in \mathbb{R}^n$ soluzione di riferimento, calcolata come qui di seguito. Utilizzando la funzione `iterative_solve` risolvere il problema sia con il metodo di Jacobi che con il metodo di Gauss-Seidel. Calcolare l'errore relativo ottenuto rispetto a $x^*$ e il numero di iterazioni effettuate. Commentare i risultati ottenuti.

Si utilizzi una tolleranza relativa pari a $1e-6$ e un numero massimo di iterazioni pari a 100.

In [ ]:
from functions import iterative_solve

# b = ...

x_star = np.linalg.solve(A, b)

# **Esercizio 2 (10 punti)**


Si consideri la seguente funzione $f$ definita nell'intervallo $I = [-\frac{3}{4}, \frac{3}{4}]$, estremi inclusi:

$$
f(x) = x \sin(2\pi x)^2
$$

#### **(2a) (1 punti) [P+T]**

Rappresentare graficamente la funzione $f$ e calcolare analiticamente i suoi zeri nell'intervallo $I$.

#### **(2b) (3 punti) [T]**

Derivare il metodo di Newton e Newton modificato per la ricerca degli zeri di una funzione, specificandone le proprietà di convergenza.

#### **(2c) (4 punti) [P]**

Date le proprietà di convergenza sopra descritte, e le proprietà degli zeri, calcolare questi ultimi utilizzahndo la funzione `newton`. Si utilizzino i punti di partenza dati da $x_0 = \alpha + \epsilon$, dove $\alpha$ è lo zero esatto e $\epsilon = 0.1$.

**Attenzione** si utilizzino in maniera *opportuna* il metodo di Newton o di Newton modificato per ottenere la più rapida convergenza.

#### **(2d) (2 punti) [P+T]**

Identificare per quale zero di $f$ è possibile applicare il metodo di bisezione, quindi usare la funzione `bisez` per calcolarlo. Si commenti il risultato ottenuto.

# **Esercizio 3 (10+1 punti)**




Si consideri il seguente problema differenziale tempo dipendente sul dominio $\Omega_T=(a,b)\times[0,T)$

$$
\begin{cases}
\partial_t u - \partial_{xx}u = f\quad \mathrm{in}\,\Omega_T\\
u(x,0) = 0 \quad \mathrm{in}\,(a,b)\\
u(a,t) = 0 \quad \mathrm{in}\,[0,T)\\
u(b,t) = 0 \quad \mathrm{in}\,[0,T),
\end{cases}
$$

con $a=0$, $b=1$, $f=1$ e $T=1$.

#### **(3a) (4 punti) [T]**

Introdurre per il problema in esame la discretizzazione in spazio con il metodo degli elementi finiti, e in tempo con il generico theta metodo.

#### **(3b) (2 punti) [P]**

Completare la funzione `heatsolve` fornita assemblando, all'interno del ciclo temporale, la matrice ed il termine noto per un generico theta.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from fem import install
from fem import xtplot

install()

In [2]:
from fem import Line, generate_mesh, FEspace, assemble, interpolate, deriv, dx, ds, DirichletBC, applyBCs, dof2fun, fun2dof, dofs, plot

def heatSolve(D,f,u0,L,h,T,dt,theta):
  """"
  Input:
     D      (float)                  Coefficiente di diffusione (positivo).
     f      (lambda function)        Forzante. Si assume f = f(x,t).
     u0     (lambda function)        Condizione iniziale.
     L      (float)                  Lunghezza dell'intervallo spaziale.
     h      (float)                  Passo della griglia spaziale.
     T      (float)                  Tempo finale
     dt     (float)                  Passo temporale.
     theta  (float)                  Parametro del theta-metodo.

  Output:
    V                               spazio elementi finiti
    u     (numpy.ndarray)-> matrix  Matrice contenente la soluzione
                                    approssimata del problema. Uij
                                    approssima u(dof_i, tj): ogni colonna è un
                                    tempo fissato.
    t      (numpy.ndarray)-> vector Griglia temporale.
  """""
  # costruisco il dominio
  domain = Line(0, L)
  # costruisco la mesh
  mesh = generate_mesh(domain, stepsize = h)
  # costruisco lo spazio FEM di grado 1
  V = FEspace(mesh, 1)

  # costruisco la griglia temporale
  nt = np.ceil(T/dt)+1
  t = np.zeros(int(nt))

  # initializzo la soluzione
  u = np.zeros((dofs(V).size, int(nt)))

  # definisco la condizione iniziale
  u0h = fun2dof(interpolate(u0,V))
  u[:, 0] = u0h

  # matrice di massa
  def m(u, v):
    return u*v*dx
  # assemblaggio matrice di massa
  M = assemble(m, V)

  # matrice di diffusione
  def a(u,v):
    return deriv(u)*deriv(v)*dx
  # assemblaggio matrice di diffusione
  A = D*assemble(a,V)

  # ciclo temporale
  for n in range(int(nt)-1):
    # costruzioni termini noti al tempo dt e dt+1
    t_old = n*dt
    t_new = (n+1)*dt

    fold = lambda x: f(x,t_old)
    fnew = lambda x: f(x,t_new)

    fold_h = interpolate(fold, V)
    def lold(v):
      return fold_h*v*dx
    Fold = assemble(lold, V)

    fnew_h = interpolate(fnew, V)
    def lnew(v):
      return fnew_h*v*dx
    Fnew = assemble(lnew, V)

    # condizioni al bordo omogenee di tipo dirichlet
    def isLeftNode(x):
      return x < 1e-12

    def isRightNode(x):
      return x > L - 1e-12

    dbc1 = DirichletBC(isLeftNode,  0.0)
    dbc2 = DirichletBC(isRightNode, 0.0)

    A = applyBCs(A, V, dbc1, dbc2)
    M = applyBCs(M, V, dbc1, dbc2)
    Fold = applyBCs(Fold, V, dbc1, dbc2)
    Fnew = applyBCs(Fnew, V, dbc1, dbc2)

    # TODO: assemblare la matrice e il termine noto per il theta metodo
    B =
    b =

    from scipy.sparse.linalg import spsolve

    u[:,n+1] = spsolve(B, b)
    t[n+1] = t_new

  return V,u,t


#### **(3c) (2 punti) [P+T]**

Risolvere il problema con il metodo di Eulero Implicito, ponendo $h=0.1$ e $dt = 0.01$. Rappresentare e commentare il risultato ottenuto.

#### **(3d) (1 punto) [P]**

Rappresentare l'andamento della soluzione in $x=0.5$ al variare del tempo.

#### **(3e) (1 punto) [P+T]**

Mantenendo gli stessi parametri di discretizzazione, ora risolvere il problema con il metodo di Eulero Esplicito e rappresentare il risultato ottenuto. Commentare il risultato ottenuto alla luce della **teoria** e proporre una strategia migliorativa.

#### **(3f) (1 punto extra) [T]**

Calcolare analiticamente la soluzione stazionaria dell'equazione.
